# 🗄️ Working with SQL Data in Google Colab
**Two methods covered:**
1. **Local .sql file** — upload `campusx_dataset.sql` and query it
2. **Live API** — fetch data from a public API and query it with SQL

---

## 📁 PART 1 — Working with a Local `.sql` File

**What is a `.sql` file?**  
It's a plain text file containing SQL commands — `CREATE TABLE` and `INSERT INTO` statements.  
We load it into SQLite (which is built into Python), and then query it using pandas.

**Steps:**
1. Upload the `campusx_dataset.sql` file to Colab
2. Create a SQLite database and run the SQL script into it
3. Explore the tables
4. Run queries

In [1]:
# ── Step 1: Upload the .sql file ─────────────────────────────────────────────
# A file picker dialog will open. Select campusx_dataset.sql from your computer.

from google.colab import files
uploaded = files.upload()

Saving campusx_dataset.sql to campusx_dataset.sql


In [2]:
# ── Step 2: Load the .sql file into a SQLite database ────────────────────────
import sqlite3

# Create a new SQLite database file
conn = sqlite3.connect("campusx.db")
cursor = conn.cursor()

# Read the uploaded .sql file
with open("campusx_dataset.sql", "r") as f:
    sql_script = f.read()

# Execute the entire script (creates tables + inserts all rows)
cursor.executescript(sql_script)
conn.commit()

print("✅ Done! Database is ready.")

✅ Done! Database is ready.


In [3]:
# ── Step 3: See what tables exist in the database ────────────────────────────
import pandas as pd

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)
print("Tables in database:")
print(tables)

Tables in database:
          name
0     students
1      courses
2  enrollments


In [4]:
# ── Step 4: Check schema (column names and types) of each table ──────────────

for table in ['students', 'courses', 'enrollments']:
    print(f"\n── Schema of '{table}' ──")
    schema = pd.read_sql_query(f"PRAGMA table_info({table})", conn)
    print(schema[['name', 'type']])


── Schema of 'students' ──
          name     type
0   student_id  INTEGER
1         name     TEXT
2          age  INTEGER
3         city     TEXT
4  enrolled_on     TEXT

── Schema of 'courses' ──
           name     type
0     course_id  INTEGER
1   course_name     TEXT
2    instructor     TEXT
3  duration_hrs  INTEGER
4      category     TEXT

── Schema of 'enrollments' ──
            name     type
0  enrollment_id  INTEGER
1     student_id  INTEGER
2      course_id  INTEGER
3          score     REAL
4          grade     TEXT
5      completed  INTEGER


In [5]:
# ── Query 1: Simple SELECT — View all students ────────────────────────────────

df = pd.read_sql_query("SELECT * FROM students", conn)
df

,student_id,name,age,city,enrolled_on
0,1,Aarav Sharma,21,Jaipur,2024-01-10
1,2,Priya Mehta,22,Delhi,2024-01-12
2,3,Rohan Verma,20,Mumbai,2024-01-15
3,4,Sneha Gupta,23,Bangalore,2024-01-18
4,5,Karan Singh,21,Jaipur,2024-02-01
5,6,Ananya Rao,22,Hyderabad,2024-02-03
6,7,Vikram Patel,24,Ahmedabad,2024-02-05
7,8,Neha Joshi,20,Pune,2024-02-10
8,9,Arjun Das,23,Kolkata,2024-02-14
9,10,Ishaan Nair,21,Kochi,2024-02-20


In [6]:
# ── Query 2: WHERE clause — Students from Jaipur ──────────────────────────────

df = pd.read_sql_query("""
    SELECT name, age, city
    FROM students
    WHERE city = 'Jaipur'
""", conn)
df

,name,age,city
0,Aarav Sharma,21,Jaipur
1,Karan Singh,21,Jaipur


In [7]:
# ── Query 3: JOIN — Which student enrolled in which course and their score ────

df = pd.read_sql_query("""
    SELECT
        s.name         AS Student,
        c.course_name  AS Course,
        e.score        AS Score,
        e.grade        AS Grade
    FROM enrollments e
    JOIN students s ON e.student_id = s.student_id
    JOIN courses  c ON e.course_id  = c.course_id
    ORDER BY e.score DESC
""", conn)
df

,Student,Course,Score,Grade
0,Siddharth Yadav,100 Days of ML,97.0,A+
1,Priya Mehta,100 Days of ML,95.0,A+
2,Siddharth Yadav,Deep Learning A-Z,94.0,A+
3,Neha Joshi,100 Days of ML,93.0,A+
4,Aarav Sharma,Python Bootcamp,92.0,A+
5,Sneha Gupta,100 Days of ML,91.5,A+
6,Ishaan Nair,SQL for Data Science,91.0,A+
7,Sneha Gupta,NLP with Python,89.0,A
8,Aarav Sharma,100 Days of ML,88.5,A
9,Vikram Patel,Feature Engineering,88.0,A


In [8]:
# ── Query 4: GROUP BY — Average score per course ──────────────────────────────

df = pd.read_sql_query("""
    SELECT
        c.course_name          AS Course,
        COUNT(e.enrollment_id) AS Total_Students,
        ROUND(AVG(e.score), 2) AS Avg_Score,
        MAX(e.score)           AS Top_Score,
        MIN(e.score)           AS Low_Score
    FROM enrollments e
    JOIN courses c ON e.course_id = c.course_id
    GROUP BY c.course_name
    ORDER BY Avg_Score DESC
""", conn)
df

,Course,Total_Students,Avg_Score,Top_Score,Low_Score
0,100 Days of ML,11,80.77,97.0,45.0
1,Deep Learning A-Z,4,79.13,94.0,62.0
2,Feature Engineering,3,78.83,88.0,65.0
3,SQL for Data Science,4,78.75,91.0,60.0
4,NLP with Python,3,78.00,89.0,68.0
5,Python Bootcamp,6,73.08,92.0,50.0


In [9]:
# ── Query 5: Subquery — Students who scored above average ─────────────────────

df = pd.read_sql_query("""
    SELECT s.name, e.score, c.course_name
    FROM enrollments e
    JOIN students s ON e.student_id = s.student_id
    JOIN courses  c ON e.course_id  = c.course_id
    WHERE e.score > (SELECT AVG(score) FROM enrollments)
    ORDER BY e.score DESC
""", conn)

print(f"Overall average score: {pd.read_sql_query('SELECT ROUND(AVG(score),2) AS avg FROM enrollments', conn).iloc[0,0]}")
df

Overall average score: 78.35


,name,score,course_name
0,Siddharth Yadav,97.0,100 Days of ML
1,Priya Mehta,95.0,100 Days of ML
2,Siddharth Yadav,94.0,Deep Learning A-Z
3,Neha Joshi,93.0,100 Days of ML
4,Aarav Sharma,92.0,Python Bootcamp
5,Sneha Gupta,91.5,100 Days of ML
6,Ishaan Nair,91.0,SQL for Data Science
7,Sneha Gupta,89.0,NLP with Python
8,Aarav Sharma,88.5,100 Days of ML
9,Vikram Patel,88.0,Feature Engineering


In [10]:
# ── Query 6: HAVING — Cities with more than 1 student ────────────────────────

df = pd.read_sql_query("""
    SELECT city, COUNT(*) AS student_count
    FROM students
    GROUP BY city
    HAVING COUNT(*) > 1
    ORDER BY student_count DESC
""", conn)
df

,city,student_count
0,Mumbai,2
1,Kolkata,2
2,Jaipur,2
3,Delhi,2


In [11]:
# ── Always close the connection when done with local DB ───────────────────────
conn.close()
print("Connection closed.")

Connection closed.


---
## 🌐 PART 2 — Working with an API Dataset

**What is an API dataset?**  
Instead of a local file, the data lives on a server. You send an HTTP request, get back JSON, convert it to a DataFrame, then query it with SQL — same as before from that point.

**API we'll use:** `restcountries.com` — free, no API key needed, returns data for every country in the world.

**Steps:**
1. Fetch JSON data from the API using `requests`
2. Convert JSON → pandas DataFrame
3. Load DataFrame into an in-memory SQLite database
4. Run SQL queries

In [12]:
# ── Step 1: Fetch data from the API ──────────────────────────────────────────
import requests

url = "https://restcountries.com/v3.1/all?fields=name,population,area,region,subregion,capital,languages"
response = requests.get(url)

print(f"Status code: {response.status_code}")  # 200 means success

raw_data = response.json()
print(f"Total countries fetched: {len(raw_data)}")
print("\nSample raw record:")
raw_data[0]  # see what one record looks like

Status code: 200
Total countries fetched: 250

Sample raw record:


{'name': {'common': 'Cook Islands',
  'official': 'Cook Islands',
  'nativeName': {'eng': {'official': 'Cook Islands', 'common': 'Cook Islands'},
   'rar': {'official': "Kūki 'Āirani", 'common': "Kūki 'Āirani"}}},
 'languages': {'eng': 'English', 'rar': 'Cook Islands Māori'},
 'capital': ['Avarua'],
 'region': 'Oceania',
 'subregion': 'Polynesia',
 'area': 236.0,
 'population': 15040}

In [13]:
# ── Step 2: Clean JSON → flat DataFrame ──────────────────────────────────────
# JSON from APIs is often nested. We flatten it into simple columns.

import pandas as pd

rows = []
for c in raw_data:
    rows.append({
        "country":    c.get("name", {}).get("common", "N/A"),
        "population": c.get("population", 0),
        "area_km2":   c.get("area", 0),
        "region":     c.get("region", "N/A"),
        "subregion":  c.get("subregion", "N/A"),
        "capital":    c.get("capital", ["N/A"])[0] if c.get("capital") else "N/A",
    })

df_countries = pd.DataFrame(rows)
print(f"Shape: {df_countries.shape}")
df_countries.head()

Shape: (250, 6)


,country,population,area_km2,region,subregion,capital
0,Cook Islands,15040,236.0,Oceania,Polynesia,Avarua
1,Guinea,14363931,245857.0,Africa,Western Africa,Conakry
2,Christmas Island,1692,135.0,Oceania,Australia and New Zealand,Flying Fish Cove
3,Togo,8095498,56785.0,Africa,Western Africa,Lomé
4,Taiwan,23317031,36197.0,Asia,Eastern Asia,Taipei


In [14]:
# ── Step 3: Load into in-memory SQLite ────────────────────────────────────────
import sqlite3

conn_api = sqlite3.connect(":memory:")  # :memory: = no file, lives in RAM only

df_countries.to_sql("countries", conn_api, index=False, if_exists="replace")

print("✅ Table 'countries' loaded into SQLite.")

✅ Table 'countries' loaded into SQLite.


In [15]:
# ── Query 1: Top 10 most populated countries ──────────────────────────────────

df = pd.read_sql_query("""
    SELECT country, population, region
    FROM countries
    ORDER BY population DESC
    LIMIT 10
""", conn_api)
df

,country,population,region
0,India,1417492000,Asia
1,China,1408280000,Asia
2,United States,340110988,Americas
3,Indonesia,284438782,Asia
4,Pakistan,241499431,Asia
5,Nigeria,223800000,Africa
6,Brazil,213421037,Americas
7,Bangladesh,169828911,Asia
8,Russia,146028325,Europe
9,Mexico,130575786,Americas


In [16]:
# ── Query 2: Filter — Countries in Asia ───────────────────────────────────────

df = pd.read_sql_query("""
    SELECT country, population, capital, subregion
    FROM countries
    WHERE region = 'Asia'
    ORDER BY population DESC
    LIMIT 15
""", conn_api)
df

,country,population,capital,subregion
0,India,1417492000,New Delhi,Southern Asia
1,China,1408280000,Beijing,Eastern Asia
2,Indonesia,284438782,Jakarta,South-Eastern Asia
3,Pakistan,241499431,Islamabad,Southern Asia
4,Bangladesh,169828911,Dhaka,Southern Asia
5,Japan,123210000,Tokyo,Eastern Asia
6,Philippines,114123600,Manila,South-Eastern Asia
7,Vietnam,101343800,Hanoi,South-Eastern Asia
8,Iran,85961000,Tehran,Southern Asia
9,Turkey,85664944,Ankara,Western Asia


In [17]:
# ── Query 3: GROUP BY — Average population per region ────────────────────────

df = pd.read_sql_query("""
    SELECT
        region,
        COUNT(*)                       AS total_countries,
        ROUND(AVG(population))         AS avg_population,
        MAX(population)                AS largest_population,
        ROUND(SUM(area_km2))           AS total_area_km2
    FROM countries
    WHERE region != 'N/A'
    GROUP BY region
    ORDER BY total_countries DESC
""", conn_api)
df

,region,total_countries,avg_population,largest_population,total_area_km2
0,Africa,59,24787532.0,223800000,30320633.0
1,Americas,56,18617496.0,340110988,42230379.0
2,Europe,53,13993546.0,146028325,23129738.0
3,Asia,50,94494639.0,1417492000,32053168.0
4,Oceania,27,1779988.0,27536874,8513684.0
5,Antarctic,5,340.0,1300,14012111.0


In [18]:
# ── Query 4: Population density — calculated column ──────────────────────────

df = pd.read_sql_query("""
    SELECT
        country,
        population,
        area_km2,
        ROUND(CAST(population AS FLOAT) / area_km2, 2) AS density_per_km2
    FROM countries
    WHERE area_km2 > 1000
    ORDER BY density_per_km2 DESC
    LIMIT 10
""", conn_api)
df

,country,population,area_km2,density_per_km2
0,Hong Kong,7527500,1104.0,6818.39
1,Bangladesh,169828911,147570.0,1150.84
2,Palestine,5483450,6220.0,881.58
3,Taiwan,23317031,36197.0,644.17
4,Mauritius,1243741,2040.0,609.68
5,Rwanda,14104969,26338.0,535.54
6,Lebanon,5490000,10452.0,525.26
7,South Korea,51159889,100210.0,510.53
8,Comoros,919901,1862.0,494.04
9,Israel,10134800,21937.0,462.00


In [19]:
# ── Close API connection ──────────────────────────────────────────────────────
conn_api.close()
print("Done!")

Done!


---
## 📌 Summary

| Method | File type | How to load | Query with |
|---|---|---|---|
| Local SQL file | `.sql` | `executescript()` | `pd.read_sql_query()` |
| Local DB file | `.db` / `.sqlite` | `connect()` directly | `pd.read_sql_query()` |
| API data | JSON from URL | `requests.get()` → `to_sql()` | `pd.read_sql_query()` |

**No XAMPP. No MySQL. Just `sqlite3` + `pandas` — that's all you need.**